# TP Jour 3 – UC-06 : transformation, qualité des données et Data Lakehouse

**Retail Intelligence Platform** · Ghassen FOUDA · 18/11/2026

## Objectif

Construire une chaîne de transformation **Raw → Silver → Gold** avec PySpark, à partir de 4 sources TPCx-BB réellement inspectées, pour répondre à deux questions métier du Jour 1 :

| Question (Jour 1) | Profil | Jeu Gold produit |
|---|---|---|
| **Q06** – Mon magasin est-il plus ou moins performant que les autres ? | Responsable de magasin | `gold/ventes_magasin_mois` |
| **Q03** – Quels produits et catégories génèrent le plus de CA et de volume ? | Direction commerciale | `gold/ventes_categorie_annee` |

## Environnement utilisé (à lire)

- Le notebook tourne dans **JupyterLab sous Ubuntu/WSL**, avec Spark en **mode local** (`local[*]`) sur une seule machine : **ce n'est pas un cluster**.
- Les fichiers sources sont lus depuis le dossier local `data/` du projet. Les traitements de ce notebook ne sont **pas exécutés sur HDFS**.
- Les fichiers sources de `data/` ne sont **jamais modifiés** : ils sont copiés dans l'espace de travail `tp3/raw/`, et tous les résultats sont écrits dans des dossiers séparés.

## Plan du notebook

0. Installation et paramètres
1. Session Spark et fiche d'environnement
2. Inventaire des sources
3. Exploration
4. Règles de nettoyage
5. Typage et nettoyage → Silver
6. Jointures vérifiées
7. Agrégations → Gold
8. Parquet : écriture, relecture, comparaison des formats
9. Rapport qualité
10. Sauvegarde et conclusion

## 0. Installation et paramètres

Cette cellule installe PySpark **seulement s'il n'est pas déjà présent** dans Colab.

In [7]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    print("PySpark installé.")
else:
    print("PySpark déjà disponible.")

PySpark déjà disponible.


Cette cellule configure les chemins du TP pour l'exécution locale dans JupyterLab sous Ubuntu/WSL.

Organisation des dossiers :
- `DOSSIER_DRIVE/data` : fichiers sources locaux (lecture seule) ;
- `DOSSIER_TRAVAIL/raw` : copie de travail des sources ;
- `silver/`, `gold/`, `quality/` (rejets et rapport qualité), `outputs/` (comparaisons, journal).


In [8]:
import os, shutil, platform, datetime, sys
from pathlib import Path

# Exécution locale dans JupyterLab sous Ubuntu/WSL.
# Le notebook peut être lancé depuis retail_project/ OU depuis retail_project/J3/.
DOSSIER_COURANT = Path.cwd().resolve()

# Remonter les dossiers parents jusqu'à trouver le vrai dossier racine contenant data/
DOSSIER_DRIVE = None
for dossier in [DOSSIER_COURANT, *DOSSIER_COURANT.parents]:
    if (dossier / "data").is_dir():
        DOSSIER_DRIVE = str(dossier)
        break

if DOSSIER_DRIVE is None:
    raise FileNotFoundError(
        f"Impossible de trouver le dossier data/. Dossier courant : {DOSSIER_COURANT}. "
        "Lance le notebook depuis retail_project ou l'un de ses sous-dossiers."
    )

DOSSIER_TRAVAIL = os.path.join(DOSSIER_DRIVE, "tp3")

# Sources retenues : nom logique -> nom réel du fichier
SOURCES = {
    "store_sales": "store_sales.dat",
    "item": "item.dat",
    "store": "store.dat",
    "date_dim": "date_dim.dat",
}

DOSSIER_RAW     = os.path.join(DOSSIER_TRAVAIL, "raw")
DOSSIER_SILVER  = os.path.join(DOSSIER_TRAVAIL, "silver")
DOSSIER_GOLD    = os.path.join(DOSSIER_TRAVAIL, "gold")
DOSSIER_QUALITY = os.path.join(DOSSIER_TRAVAIL, "quality")
DOSSIER_OUTPUTS = os.path.join(DOSSIER_TRAVAIL, "outputs")
VERSION_REGLES  = "v1"
DATE_TRAITEMENT = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Vérifier explicitement les quatre sources attendues avant toute copie.
for table, fichier in SOURCES.items():
    chemin_source = os.path.join(DOSSIER_DRIVE, "data", table, fichier)
    if not os.path.isfile(chemin_source):
        raise FileNotFoundError(f"Fichier source introuvable : {chemin_source}")

for d in [DOSSIER_RAW, DOSSIER_SILVER, DOSSIER_GOLD, DOSSIER_QUALITY, DOSSIER_OUTPUTS]:
    os.makedirs(d, exist_ok=True)

# Journal d'exécution (preuve de fonctionnement)
JOURNAL = []
def journal(message):
    ligne = f"{datetime.datetime.now():%H:%M:%S} | {message}"
    JOURNAL.append(ligne)
    print(ligne)

# Copie des sources : les originaux dans data/ ne sont jamais modifiés.
for table, fichier in SOURCES.items():
    source = os.path.join(DOSSIER_DRIVE, "data", table, fichier)
    shutil.copy2(source, os.path.join(DOSSIER_RAW, fichier))
    journal(f"Copie de {fichier} ({os.path.getsize(source):,} octets)".replace(",", " "))

print("\nRacine du projet détectée :", DOSSIER_DRIVE)
print("Dossier de travail :", DOSSIER_TRAVAIL)


15:02:19 | Copie de store_sales.dat (92 764 700 octets)
15:02:20 | Copie de item.dat (6 799 286 octets)
15:02:20 | Copie de store.dat (3 386 octets)
15:02:20 | Copie de date_dim.dat (15 395 800 octets)

Racine du projet détectée : /mnt/c/Users/found/desktop/ipssi/bd/retail_project/J3
Dossier de travail : /mnt/c/Users/found/desktop/ipssi/bd/retail_project/J3/tp3


## 1. Session Spark et fiche d'environnement

On crée **une seule** session Spark, réutilisée dans tout le notebook.

- `master("local[*]")` : Spark s'exécute sur la machine Colab, en utilisant tous ses processeurs.
- `spark.sql.ansi.enabled = false` : une valeur impossible à convertir (par exemple du texte dans une colonne numérique) devient `null` au lieu d'arrêter le traitement. On peut ainsi **compter** ces échecs de conversion et les traiter comme une anomalie.

In [10]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 21.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 24.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [13]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
import pandas as pd

# Utiliser explicitement le système de fichiers local
DOSSIER_WAREHOUSE = Path(DOSSIER_TRAVAIL, "spark-warehouse").resolve().as_uri()

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .config("spark.sql.warehouse.dir", DOSSIER_WAREHOUSE)
    .getOrCreate()
)

# Forcer également la configuration Hadoop de la session active
spark.sparkContext._jsc.hadoopConfiguration().set(
    "fs.defaultFS", "file:///"
)

print("Spark version:", spark.version)
print(
    "Filesystem:",
    spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS")
)

def version_java():
    r = subprocess.run(["java", "-version"], capture_output=True, text=True)
    lignes = [l for l in r.stderr.splitlines() if "version" in l]
    return lignes[0] if lignes else "inconnue"

def memoire_go():
    try:
        with open("/proc/meminfo") as f:
            return round(int(f.readline().split()[1]) / 1024 / 1024, 1)
    except Exception:
        return "inconnue"

fiche_env = pd.DataFrame([
    ("Plateforme", "Google Colab" if EN_COLAB else platform.platform()),
    ("Python", platform.python_version()),
    ("Spark / PySpark", spark.version),
    ("Java", version_java()),
    ("Mode d'exécution", spark.sparkContext.master + " (local, une seule machine)"),
    ("Processeurs", os.cpu_count()),
    ("Mémoire (Go)", memoire_go()),
    ("Données sources", f"{DOSSIER_DRIVE}/data"),
    ("Dossier de travail", DOSSIER_TRAVAIL),
    ("Date du traitement", DATE_TRAITEMENT),
], columns=["Élément", "Valeur observée"])
fiche_env.to_csv(f"{DOSSIER_OUTPUTS}/fiche_environnement.csv", index=False)
journal("Session Spark créée")
fiche_env

Spark version: 4.2.1
Filesystem: file:///
15:25:46 | Session Spark créée


,Élément,Valeur observée
0,Plateforme,Linux-6.18.58.1-microsoft-standard-WSL2-x86_64...
1,Python,3.10.12
2,Spark / PySpark,4.2.1
3,Java,"openjdk version ""21.0.13.0.3"" 2026-10-24"
4,Mode d'exécution,"local[*] (local, une seule machine)"
5,Processeurs,8
6,Mémoire (Go),7.7
7,Données sources,/mnt/c/Users/found/desktop/ipssi/bd/retail_pro...
8,Dossier de travail,/mnt/c/Users/found/desktop/ipssi/bd/retail_pro...
9,Date du traitement,2026-11-18 15:02:17


### Test rapide de lecture et d'écriture

Vérifications attendues à l'étape 1 du TP : lire un petit jeu de test, afficher son schéma, puis écrire un résultat dans un répertoire autorisé.

In [14]:
df_test = spark.createDataFrame([("Magasin A", 1200.0), ("Magasin B", 850.0)], ["magasin", "chiffre_affaires"])
df_test.printSchema()
df_test.write.mode("overwrite").parquet(f"{DOSSIER_OUTPUTS}/test_ecriture")
print("Relecture du test :", spark.read.parquet(f"{DOSSIER_OUTPUTS}/test_ecriture").count(), "lignes")

root
 |-- magasin: string (nullable = true)
 |-- chiffre_affaires: double (nullable = true)



Relecture du test : 2 lignes


## 2. Inventaire des sources

**Format constaté au Jour 1 :** fichiers texte `.dat`, séparateur `|`, **sans ligne d'en-tête**, fins de ligne Windows (CRLF).

Les noms de colonnes ne figurent pas dans les fichiers. Ils proviennent du **schéma de référence TPCx-BB** (statut : *déduit*, comme au Jour 1). On contrôle donc que **chaque ligne contient exactement le nombre de colonnes attendu**.

Choix de lecture : on lit d'abord **toutes les colonnes en texte** (sans `inferSchema`), puis on convertit chaque colonne **explicitement** à l'étape 5. L'inférence automatique peut se tromper (un identifiant lu comme un nombre décimal, un code postal sans ses zéros…), et une conversion explicite est **contrôlable**.

In [15]:
COLONNES = {
    "store_sales": "ss_sold_date_sk ss_sold_time_sk ss_item_sk ss_customer_sk ss_cdemo_sk ss_hdemo_sk ss_addr_sk ss_store_sk ss_promo_sk ss_ticket_number ss_quantity ss_wholesale_cost ss_list_price ss_sales_price ss_ext_discount_amt ss_ext_sales_price ss_ext_wholesale_cost ss_ext_list_price ss_ext_tax ss_coupon_amt ss_net_paid ss_net_paid_inc_tax ss_net_profit".split(),
    "item": "i_item_sk i_item_id i_rec_start_date i_rec_end_date i_item_desc i_current_price i_wholesale_cost i_brand_id i_brand i_class_id i_class i_category_id i_category i_manufact_id i_manufact i_size i_formulation i_color i_units i_container i_manager_id i_product_name".split(),
    "store": "s_store_sk s_store_id s_rec_start_date s_rec_end_date s_closed_date_sk s_store_name s_number_employees s_floor_space s_hours s_manager s_market_id s_geography_class s_market_desc s_market_manager s_division_id s_division_name s_company_id s_company_name s_street_number s_street_name s_street_type s_suite_number s_city s_county s_state s_zip s_country s_gmt_offset s_tax_precentage".split(),
    "date_dim": "d_date_sk d_date_id d_date d_month_seq d_week_seq d_quarter_seq d_year d_dow d_moy d_dom d_qoy d_fy_year d_fy_quarter_seq d_fy_week_seq d_day_name d_quarter_name d_holiday d_weekend d_following_holiday d_first_dom d_last_dom d_same_day_ly d_same_day_lq d_current_day d_current_week d_current_month d_current_quarter d_current_year".split(),
}

# Ce que représente une ligne (granularité) et clé candidate
GRAIN = {
    "store_sales": ("une ligne = un article vendu sur un ticket de caisse", "ss_ticket_number + ss_item_sk"),
    "item": ("une ligne = une version d'un article", "i_item_sk"),
    "store": ("une ligne = une version d'un magasin", "s_store_sk"),
    "date_dim": ("une ligne = un jour du calendrier", "d_date_sk"),
}

def lire_brut(table):
    """Lit un fichier .dat : toutes les colonnes en texte, noms issus du schéma de référence."""
    schema = T.StructType([T.StructField(c, T.StringType(), True) for c in COLONNES[table]])
    return (spark.read
            .option("sep", "|")
            .option("header", False)
            .option("encoding", "UTF-8")
            .schema(schema)
            .csv(f"{DOSSIER_RAW}/{SOURCES[table]}"))

brut = {t: lire_brut(t) for t in SOURCES}

inventaire = []
for table, fichier in SOURCES.items():
    chemin = f"{DOSSIER_RAW}/{fichier}"
    lignes_txt = spark.read.text(chemin)
    nb_champs = (lignes_txt
                 .select(F.size(F.split(F.regexp_replace("value", "\r$", ""), r"\|")).alias("n"))
                 .groupBy("n").count().collect())
    inventaire.append({
        "Source": table,
        "Fichier réel": fichier,
        "Emplacement": os.path.join(DOSSIER_DRIVE, "data", table, fichier),
        "Format": "texte .dat, séparateur |, sans en-tête",
        "Taille (Mo)": round(os.path.getsize(chemin) / 1024 / 1024, 1),
        "Lignes": brut[table].count(),
        "Colonnes attendues": len(COLONNES[table]),
        "Colonnes par ligne (observé)": ", ".join(f"{r['n']} ({r['count']} lignes)" for r in nb_champs),
        "Granularité": GRAIN[table][0],
        "Clé candidate": GRAIN[table][1],
    })

df_inventaire = pd.DataFrame(inventaire)
df_inventaire.to_csv(f"{DOSSIER_OUTPUTS}/inventaire_sources.csv", index=False)
journal("Inventaire des sources réalisé")
df_inventaire

15:26:10 | Inventaire des sources réalisé


,Source,Fichier réel,Emplacement,Format,Taille (Mo),Lignes,Colonnes attendues,Colonnes par ligne (observé),Granularité,Clé candidate
0,store_sales,store_sales.dat,/mnt/c/Users/found/desktop/ipssi/bd/retail_pro...,"texte .dat, séparateur |, sans en-tête",88.5,667579,23,23 (667579 lignes),une ligne = un article vendu sur un ticket de ...,ss_ticket_number + ss_item_sk
1,item,item.dat,/mnt/c/Users/found/desktop/ipssi/bd/retail_pro...,"texte .dat, séparateur |, sans en-tête",6.5,17820,22,22 (17820 lignes),une ligne = une version d'un article,i_item_sk
2,store,store.dat,/mnt/c/Users/found/desktop/ipssi/bd/retail_pro...,"texte .dat, séparateur |, sans en-tête",0.0,12,29,29 (12 lignes),une ligne = une version d'un magasin,s_store_sk
3,date_dim,date_dim.dat,/mnt/c/Users/found/desktop/ipssi/bd/retail_pro...,"texte .dat, séparateur |, sans en-tête",14.7,109573,28,28 (109573 lignes),une ligne = un jour du calendrier,d_date_sk


**À vérifier :** pour chaque source, toutes les lignes ont le nombre de colonnes attendu (une seule valeur dans « Colonnes par ligne »). Les nombres de lignes doivent correspondre à l'inventaire du Jour 1 : `store_sales` 667 579, `item` 17 820, `store` 12, `date_dim` 109 573.

## 3. Exploration

Pour chaque source : schéma, échantillon, nombre de lignes, **valeurs manquantes par colonne** et **doublons**.

Le comptage des valeurs nulles se fait en **un seul passage** sur les données : on additionne, pour chaque colonne, 1 quand la valeur est nulle. C'est plus rapide qu'un `filter(...).count()` par colonne.

In [16]:
def valeurs_manquantes(df):
    """Nombre et pourcentage de valeurs nulles par colonne (un seul passage)."""
    total = df.count()
    ligne = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    res = pd.DataFrame({"colonne": df.columns, "nb_nulls": [ligne[c] for c in df.columns]})
    res["pct_nulls"] = (100 * res["nb_nulls"] / total).round(2)
    return res

def doublons(df, cles=None):
    """Lignes en trop : doublons exacts, ou doublons sur une clé."""
    cols = cles or df.columns
    return df.count() - df.select(*cols).distinct().count()

explorations = {}
for table in SOURCES:
    print(f"\n==================== {table} ====================")
    brut[table].printSchema()
    brut[table].show(3, truncate=40)
    vm = valeurs_manquantes(brut[table])
    explorations[table] = vm
    print("Colonnes avec des valeurs manquantes :")
    print(vm[vm.nb_nulls > 0].to_string(index=False) if (vm.nb_nulls > 0).any() else "  aucune")
    print("Doublons exacts :", doublons(brut[table]))


==================== store_sales ====================
root
 |-- ss_sold_date_sk: string (nullable = true)
 |-- ss_sold_time_sk: string (nullable = true)
 |-- ss_item_sk: string (nullable = true)
 |-- ss_customer_sk: string (nullable = true)
 |-- ss_cdemo_sk: string (nullable = true)
 |-- ss_hdemo_sk: string (nullable = true)
 |-- ss_addr_sk: string (nullable = true)
 |-- ss_store_sk: string (nullable = true)
 |-- ss_promo_sk: string (nullable = true)
 |-- ss_ticket_number: string (nullable = true)
 |-- ss_quantity: string (nullable = true)
 |-- ss_wholesale_cost: string (nullable = true)
 |-- ss_list_price: string (nullable = true)
 |-- ss_sales_price: string (nullable = true)
 |-- ss_ext_discount_amt: string (nullable = true)
 |-- ss_ext_sales_price: string (nullable = true)
 |-- ss_ext_wholesale_cost: string (nullable = true)
 |-- ss_ext_list_price: string (nullable = true)
 |-- ss_ext_tax: string (nullable = true)
 |-- ss_coupon_amt: string (nullable = true)
 |-- ss_net_paid: strin

Colonnes avec des valeurs manquantes :
    colonne  nb_nulls  pct_nulls
ss_cdemo_sk       238       0.04
ss_hdemo_sk       295       0.04
 ss_addr_sk       448       0.07
ss_promo_sk       158       0.02


Doublons exacts : 0

==================== item ====================
root
 |-- i_item_sk: string (nullable = true)
 |-- i_item_id: string (nullable = true)
 |-- i_rec_start_date: string (nullable = true)
 |-- i_rec_end_date: string (nullable = true)
 |-- i_item_desc: string (nullable = true)
 |-- i_current_price: string (nullable = true)
 |-- i_wholesale_cost: string (nullable = true)
 |-- i_brand_id: string (nullable = true)
 |-- i_brand: string (nullable = true)
 |-- i_class_id: string (nullable = true)
 |-- i_class: string (nullable = true)
 |-- i_category_id: string (nullable = true)
 |-- i_category: string (nullable = true)
 |-- i_manufact_id: string (nullable = true)
 |-- i_manufact: string (nullable = true)
 |-- i_size: string (nullable = true)
 |-- i_formulation: string (nullable = true)
 |-- i_color: string (nullable = true)
 |-- i_units: string (nullable = true)
 |-- i_container: string (nullable = true)
 |-- i_manager_id: string (nullable = true)
 |-- i_product_name: string (

[Stage 87:==============>                                           (1 + 3) / 4]

Doublons exacts : 0


### Contrôles spécifiques

- **Doublons sur la clé candidate** de chaque table. Attention au **grain** : dans `store_sales`, le même article peut apparaître plusieurs fois le même jour (ventes différentes). Le vrai doublon, c'est la même paire **ticket + article**.
- **Valeurs atypiques** sur les quantités et les montants de `store_sales` (minimum, maximum, valeurs négatives).

In [17]:
print("Doublons sur la clé candidate :")
print("  store_sales (ticket + article) :", doublons(brut["store_sales"], ["ss_ticket_number", "ss_item_sk"]))
print("  item (i_item_sk)               :", doublons(brut["item"], ["i_item_sk"]))
print("  store (s_store_sk)             :", doublons(brut["store"], ["s_store_sk"]))
print("  date_dim (d_date_sk)           :", doublons(brut["date_dim"], ["d_date_sk"]))

mesures = ["ss_quantity", "ss_sales_price", "ss_ext_sales_price", "ss_net_paid", "ss_net_profit"]
ss_num = brut["store_sales"].select([F.col(c).cast("double").alias(c) for c in mesures])
stats = ss_num.select(
    *[F.min(c).alias(f"{c}|min") for c in mesures],
    *[F.max(c).alias(f"{c}|max") for c in mesures],
    *[F.sum((F.col(c) < 0).cast("int")).alias(f"{c}|nb_negatifs") for c in mesures],
).collect()[0].asDict()
df_stats = pd.DataFrame([{"mesure": m, "min": stats[f"{m}|min"], "max": stats[f"{m}|max"],
                          "nb_negatifs": stats[f"{m}|nb_negatifs"]} for m in mesures])
df_stats

Doublons sur la clé candidate :


  store_sales (ticket + article) : 4136
  item (i_item_sk)               : 0
  store (s_store_sk)             : 0
  date_dim (d_date_sk)           : 0


,mesure,min,max,nb_negatifs
0,ss_quantity,1.0,100.0,0
1,ss_sales_price,0.0,199.3,0
2,ss_ext_sales_price,0.0,18745.0,0
3,ss_net_paid,0.0,18745.0,0
4,ss_net_profit,-9866.6,9134.0,495971


In [18]:
# Période couverte par les ventes, et lien entre une ligne de vente et un ticket
periode = (brut["store_sales"].alias("s")
           .join(brut["date_dim"].alias("d"), F.col("s.ss_sold_date_sk") == F.col("d.d_date_sk"), "left")
           .agg(F.min("d.d_date").alias("premiere_vente"), F.max("d.d_date").alias("derniere_vente"),
                F.countDistinct("s.ss_ticket_number").alias("nb_tickets")))
periode.show()
print("Lignes de vente par ticket (moyenne) :", round(brut["store_sales"].count() / periode.first()["nb_tickets"], 2))

print("Catégories d'articles :")
brut["item"].groupBy("i_category").count().orderBy("i_category").show(truncate=False)

+--------------+--------------+----------+
|premiere_vente|derniere_vente|nb_tickets|
+--------------+--------------+----------+
|    2001-01-01|    2005-12-13|     89100|
+--------------+--------------+----------+



Lignes de vente par ticket (moyenne) : 7.49
Catégories d'articles :
+------------------------+-----+
|i_category              |count|
+------------------------+-----+
|Books                   |5335 |
|Clothing & Accessories  |610  |
|Electronics             |692  |
|Home & Kitchen          |476  |
|Movies & TV             |1286 |
|Music                   |8143 |
|Sports & Outdoors       |465  |
|Tools & Home Improvement|408  |
|Toys & Games            |405  |
+------------------------+-----+



### Mes observations (exploration)

> 📝 *À compléter avec tes propres mots à partir des résultats affichés.*
>
> - Quelles colonnes sont des **identifiants** (`_sk`, `_id`), des **mesures** (quantités, prix, montants) et des **dates** ?
> - Quels champs ont des **valeurs manquantes**, et en quelle proportion ?
> - Y a-t-il des **doublons** sur la clé candidate ?
> - Quelles valeurs sont **atypiques** ? Par exemple `ss_net_profit` négatif sur une grande partie des lignes : est-ce une erreur, ou un cas métier possible (vente à perte, promotion) ?
> - Rappel : une ligne de `store_sales` est un **article sur un ticket**, pas une transaction complète.

## 4. Règles de nettoyage

Chaque règle est **explicite, mesurable et justifiée**. Aucune ligne n'est supprimée en silence : les lignes écartées sont **mises en quarantaine** dans `quality/rejets_store_sales`, avec le motif du rejet.

| N° | Anomalie | Règle | Action appliquée | Justification |
|---|---|---|---|---|
| R1 | **Type incorrect** | Chaque colonne est convertie vers le type attendu (identifiants → entier long, montants → décimal 12,2, quantités → entier, dates → date) | Conversion explicite, puis comptage des valeurs non convertibles | L'inférence automatique ne valide pas un schéma ; une conversion ratée devient `null` et doit être comptée |
| R2 | **Valeur manquante sur un champ obligatoire** | Une vente doit avoir une date, un article, un magasin et un numéro de ticket | Ligne mise en quarantaine (motif `cle_obligatoire_absente`) | Sans ces clés, la vente ne peut pas être rattachée à un magasin, un produit ou une période |
| R3 | **Valeur manquante sur un champ facultatif** (promotion, client, adresse, données démographiques) | Valeur conservée telle quelle (`null`) | Aucune | Absence légitime possible (vente sans promotion, client non identifié) ; ces champs ne servent pas aux KPI produits |
| R4 | **Doublon exact** | Deux lignes strictement identiques | Une seule conservée | Une même ligne de ticket ne peut pas être vendue deux fois |
| R5 | **Doublon sur la clé ticket + article** | La paire devrait être unique si le grain est « un article par ticket » | Contrôle, puis **analyse** des lignes concernées ; **pas de suppression automatique** | Vérifier le grain avant les agrégations ; deux lignes qui ne sont pas identiques ne sont pas forcément des doublons |
| R6 | **Quantité invalide** | `ss_quantity` > 0 | Quarantaine (motif `quantite_invalide`) | Une ligne de vente porte au moins un article ; les retours sont dans une autre table (`store_returns`) |
| R7 | **Montant négatif** | `ss_sales_price` ≥ 0 et `ss_net_paid` ≥ 0 | Quarantaine pour investigation (motif `montant_negatif`) | Un montant négatif peut être une correction : on ne le supprime pas, on l'isole |
| R8 | **Valeur atypique : marge négative** | `ss_net_profit` < 0 | **Conservée** et signalée comme limite | Cas métier possible (vente à perte, remise) ; donnée synthétique : ne pas l'interpréter comme une réalité économique |
| R9 | **Incohérence de calcul** | `ss_ext_sales_price` = `ss_quantity` × `ss_sales_price` (écart ≤ 0,01) | Contrôle ; signalement si écart | Vérifie la cohérence interne des montants |
| R10 | **Identifiant invalide (intégrité)** | `ss_item_sk` existe dans `item`, `ss_store_sk` dans `store`, `ss_sold_date_sk` dans `date_dim` | Quarantaine (motifs `article_inconnu`, `magasin_inconnu`, `date_inconnue`) | Une vente orpheline fausserait les agrégats par produit, magasin ou période |
| R11 | **Texte mal formaté** (référentiels) | Espaces superflus retirés (`trim`) | Nettoyage des colonnes texte de `item` et `store` | Harmonisation des libellés |
| R12 | **Catégorie absente** (`item`) | `i_category` manquante | Remplacée par `Non renseignée` | Conserver ces articles dans les analyses par catégorie, sans inventer de catégorie |
| R13 | **Date incohérente** (`date_dim`) | `d_date` convertible en date, `d_year`/`d_moy` cohérents avec `d_date` | Contrôle | Garantit des regroupements corrects par mois et par année |

*Les nombres de lignes concernées par chaque règle sont calculés à l'étape 5 et repris dans le rapport qualité (étape 9).*

## 5. Typage et nettoyage → zone Silver

### 5.1 Conversion explicite des types (R1)

Pour chaque colonne convertie, on compte les **échecs de conversion** : la valeur d'origine n'est pas nulle, mais la valeur convertie l'est.

In [19]:
DEC = T.DecimalType(12, 2)

def types_cibles(table):
    cols = COLONNES[table]
    types = {c: T.StringType() for c in cols}
    for c in cols:
        if c.endswith("_sk"):
            types[c] = T.LongType()
    if table == "store_sales":
        types["ss_ticket_number"] = T.LongType()
        types["ss_quantity"] = T.IntegerType()
        for c in cols[11:]:
            types[c] = DEC
    elif table == "item":
        for c in ["i_rec_start_date", "i_rec_end_date"]: types[c] = T.DateType()
        for c in ["i_current_price", "i_wholesale_cost"]: types[c] = DEC
        for c in ["i_brand_id", "i_class_id", "i_category_id", "i_manufact_id", "i_manager_id"]: types[c] = T.IntegerType()
    elif table == "store":
        for c in ["s_rec_start_date", "s_rec_end_date"]: types[c] = T.DateType()
        for c in ["s_number_employees", "s_floor_space", "s_market_id", "s_division_id", "s_company_id"]: types[c] = T.IntegerType()
        for c in ["s_gmt_offset", "s_tax_precentage"]: types[c] = T.DecimalType(5, 2)
    elif table == "date_dim":
        types["d_date"] = T.DateType()
        for c in ["d_month_seq", "d_week_seq", "d_quarter_seq", "d_year", "d_dow", "d_moy", "d_dom",
                  "d_qoy", "d_fy_year", "d_fy_quarter_seq", "d_fy_week_seq"]:
            types[c] = T.IntegerType()
        for c in ["d_first_dom", "d_last_dom", "d_same_day_ly", "d_same_day_lq"]: types[c] = T.LongType()
    return types

def typer(table):
    df = brut[table]
    types = types_cibles(table)
    a_convertir = [c for c, t in types.items() if not isinstance(t, T.StringType)]
    exprs = [(F.col(c).cast(t) if not isinstance(t, T.StringType) else F.col(c)).alias(c) for c, t in types.items()]
    echecs_expr = [F.sum((F.col(c).isNotNull() & F.col(c).cast(types[c]).isNull()).cast("int")).alias(c) for c in a_convertir]
    echecs = df.select(echecs_expr).collect()[0].asDict()
    return df.select(exprs), {c: n for c, n in echecs.items() if n}

typees, echecs_conversion = {}, {}
for table in SOURCES:
    typees[table], echecs_conversion[table] = typer(table)
    print(f"{table:12s} : {len(types_cibles(table)) - sum(isinstance(t, T.StringType) for t in types_cibles(table).values())} colonnes converties, "
          f"échecs de conversion : {echecs_conversion[table] or 'aucun'}")
typees["store_sales"].printSchema()

store_sales  : 23 colonnes converties, échecs de conversion : aucun
item         : 10 colonnes converties, échecs de conversion : aucun
store        : 11 colonnes converties, échecs de conversion : aucun
date_dim     : 17 colonnes converties, échecs de conversion : aucun
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(12,2) (nullable = true)
 |-- ss_list_price: decimal(12,2) (nullable = true)
 |-- ss_sales_price: decimal(12,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(12,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(12,

### 5.2 Nettoyage des référentiels : `item`, `store`, `date_dim` (R11, R12, R13)

In [20]:
def trim_texte(df):
    return df.select([F.trim(F.col(c)).alias(c) if isinstance(df.schema[c].dataType, T.StringType) else F.col(c)
                      for c in df.columns])

def ajouter_provenance(df, table):
    return (df.withColumn("_source_fichier", F.lit(SOURCES[table]))
              .withColumn("_date_traitement", F.lit(DATE_TRAITEMENT))
              .withColumn("_version_regles", F.lit(VERSION_REGLES)))

# item : trim + catégorie manquante
item_trim = trim_texte(typees["item"])
nb_categorie_absente = item_trim.filter(F.col("i_category").isNull() | (F.col("i_category") == "")).count()
silver_item = ajouter_provenance(
    item_trim.withColumn("i_category", F.when(F.col("i_category").isNull() | (F.col("i_category") == ""), "Non renseignée")
                                        .otherwise(F.col("i_category"))), "item")

# store : trim
silver_store = ajouter_provenance(trim_texte(typees["store"]), "store")

# date_dim : contrôle de cohérence entre d_date, d_year et d_moy
dd = typees["date_dim"]
nb_dates_incoherentes = dd.filter(
    F.col("d_date").isNull() | (F.year("d_date") != F.col("d_year")) | (F.month("d_date") != F.col("d_moy"))
).count()
silver_date = ajouter_provenance(dd, "date_dim")

print("item     : catégories absentes remplacées :", nb_categorie_absente)
print("date_dim : dates incohérentes             :", nb_dates_incoherentes)

item     : catégories absentes remplacées : 0
date_dim : dates incohérentes             : 0


### 5.3 Nettoyage de `store_sales` (R2 à R10)

On calcule pour chaque ligne un **motif de rejet**. Le premier motif rencontré l'emporte. Les lignes sans motif vont en **Silver** ; les autres vont en **quarantaine** dans `quality/`.

Les contrôles d'intégrité (R10) utilisent des **jointures gauches** vers les référentiels : si la clé ne trouve pas de correspondance, la colonne du référentiel est `null`. On vérifie juste avant que les clés des référentiels sont **uniques**, sinon la jointure dupliquerait des ventes.

In [21]:
ss = typees["store_sales"]
nb_initial = ss.count()

# R4 : doublons exacts
ss_dedup = ss.dropDuplicates()
nb_doublons_exacts = nb_initial - ss_dedup.count()

# Unicité des clés des référentiels (pré-requis des jointures)
for nom, df_ref, cle in [("item", typees["item"], "i_item_sk"), ("store", typees["store"], "s_store_sk"),
                         ("date_dim", typees["date_dim"], "d_date_sk")]:
    assert df_ref.count() == df_ref.select(cle).distinct().count(), f"Clé {cle} non unique dans {nom}"
print("Clés des référentiels uniques : OK")

ref_item  = typees["item"].select(F.col("i_item_sk").alias("_ref_item"))
ref_store = typees["store"].select(F.col("s_store_sk").alias("_ref_store"))
ref_date  = typees["date_dim"].select(F.col("d_date_sk").alias("_ref_date"))

ss_ctrl = (ss_dedup
           .join(ref_item,  F.col("ss_item_sk") == F.col("_ref_item"), "left")
           .join(ref_store, F.col("ss_store_sk") == F.col("_ref_store"), "left")
           .join(ref_date,  F.col("ss_sold_date_sk") == F.col("_ref_date"), "left"))

motif = (F.when(F.col("ss_sold_date_sk").isNull() | F.col("ss_item_sk").isNull() |
                F.col("ss_store_sk").isNull() | F.col("ss_ticket_number").isNull(), "cle_obligatoire_absente")
          .when(F.col("ss_quantity").isNull() | (F.col("ss_quantity") <= 0), "quantite_invalide")
          .when((F.col("ss_sales_price") < 0) | (F.col("ss_net_paid") < 0), "montant_negatif")
          .when(F.col("_ref_item").isNull(), "article_inconnu")
          .when(F.col("_ref_store").isNull(), "magasin_inconnu")
          .when(F.col("_ref_date").isNull(), "date_inconnue"))

ss_ctrl = ss_ctrl.withColumn("motif_rejet", motif).drop("_ref_item", "_ref_store", "_ref_date").cache()

rejets = ss_ctrl.filter(F.col("motif_rejet").isNotNull())
silver_sales = ajouter_provenance(ss_ctrl.filter(F.col("motif_rejet").isNull()).drop("motif_rejet"), "store_sales")

# R5 : unicité du grain ticket + article dans Silver
nb_doublons_grain = silver_sales.count() - silver_sales.select("ss_ticket_number", "ss_item_sk").distinct().count()

# R8 et R9 : contrôles signalés, lignes conservées
nb_marge_negative = silver_sales.filter(F.col("ss_net_profit") < 0).count()
nb_incoherence_calcul = silver_sales.filter(
    F.abs(F.col("ss_ext_sales_price") - F.col("ss_quantity") * F.col("ss_sales_price")) > 0.01).count()

nb_silver, nb_rejets = silver_sales.count(), rejets.count()
print(f"Lignes initiales        : {nb_initial:>9,}".replace(",", " "))
print(f"Doublons exacts retirés : {nb_doublons_exacts:>9,}".replace(",", " "))
print(f"Lignes en quarantaine   : {nb_rejets:>9,}".replace(",", " "))
print(f"Lignes en Silver        : {nb_silver:>9,}".replace(",", " "))
print("Contrôle des volumes    :", "OK" if nb_initial == nb_doublons_exacts + nb_rejets + nb_silver else "ÉCART")
print("Doublons ticket+article :", nb_doublons_grain)
print("Marges négatives (R8)   :", nb_marge_negative, f"({100*nb_marge_negative/nb_silver:.1f} %) – conservées")
print("Incohérences de calcul  :", nb_incoherence_calcul)
print("\nRépartition des rejets par motif :")
rejets.groupBy("motif_rejet").count().show()

Clés des référentiels uniques : OK


Lignes initiales        :   667 579
Doublons exacts retirés :         0
Lignes en quarantaine   :         0
Lignes en Silver        :   667 579
Contrôle des volumes    : OK
Doublons ticket+article : 4136
Marges négatives (R8)   : 495971 (74.3 %) – conservées
Incohérences de calcul  : 0

Répartition des rejets par motif :
+-----------+-----+
|motif_rejet|count|
+-----------+-----+
+-----------+-----+



[Stage 247:=============================================>       (172 + 9) / 200]

### 5.4 Analyse de la règle R5 : ticket + article non unique

Le contrôle R5 trouve des paires **ticket + article** présentes plusieurs fois. Avant de décider quoi que ce soit, on regarde ces lignes de près.

In [22]:
paires = silver_sales.groupBy("ss_ticket_number", "ss_item_sk").count().filter("count > 1")
print("Paires ticket + article répétées :", paires.count())
print("Répartition (nombre d'occurrences -> nombre de paires) :")
paires.groupBy(F.col("count").alias("occurrences")).count().orderBy("occurrences").show()

exemple = paires.orderBy("ss_ticket_number", "ss_item_sk").first()
(silver_sales
 .filter((F.col("ss_ticket_number") == exemple["ss_ticket_number"]) & (F.col("ss_item_sk") == exemple["ss_item_sk"]))
 .select("ss_sold_date_sk", "ss_sold_time_sk", "ss_store_sk", "ss_customer_sk", "ss_ticket_number",
         "ss_item_sk", "ss_quantity", "ss_sales_price", "ss_net_paid")
 .show())

Paires ticket + article répétées : 4038
Répartition (nombre d'occurrences -> nombre de paires) :


+-----------+-----+
|occurrences|count|
+-----------+-----+
|          2| 3942|
|          3|   94|
|          4|    2|
+-----------+-----+



+---------------+---------------+-----------+--------------+----------------+----------+-----------+--------------+-----------+
|ss_sold_date_sk|ss_sold_time_sk|ss_store_sk|ss_customer_sk|ss_ticket_number|ss_item_sk|ss_quantity|ss_sales_price|ss_net_paid|
+---------------+---------------+-----------+--------------+----------------+----------+-----------+--------------+-----------+
|          36890|          27946|          5|         92924|              44|     10003|         46|          1.60|       5.89|
|          36890|          27946|          5|         92924|              44|     10003|         31|         53.16|    1647.96|
+---------------+---------------+-----------+--------------+----------------+----------+-----------+--------------+-----------+



**Décision R5 :** les lignes répétées ont le même ticket, le même article, la même date et le même client, mais **des quantités et des prix différents**. Ce ne sont donc **pas des copies** (il y a 0 doublon exact, règle R4). Elles peuvent correspondre à deux passages en caisse du même article à des prix différents, par exemple avec et sans remise. Le fichier ne contient pas de numéro de ligne de ticket qui permettrait de les distinguer.

- **Action :** lignes **conservées**, aucune suppression.
- **Conséquence :** le grain réel de `store_sales` est « une **ligne** de ticket », et la paire ticket + article n'est **pas** une clé unique. Le nombre de transactions (KPI-03) est calculé avec `countDistinct(ss_ticket_number)`, il n'est donc pas faussé.
- **Statut :** « À surveiller », documenté comme limite dans le rapport qualité.

### 5.5 Écriture de la zone Silver (Parquet)

**Convention de nommage :** `silver/<nom de la table source>`, au format Parquet. Les colonnes gardent leur nom TPCx-BB, et trois colonnes de **provenance** sont ajoutées : `_source_fichier`, `_date_traitement`, `_version_regles`.

Le mode `overwrite` remplace le contenu de la destination. Il est utilisé **uniquement sur les dossiers de sortie du TP**, jamais sur les sources.

In [23]:
silver = {"store_sales": silver_sales, "item": silver_item, "store": silver_store, "date_dim": silver_date}
for nom, df in silver.items():
    df.write.mode("overwrite").parquet(f"{DOSSIER_SILVER}/{nom}")
    journal(f"Silver écrit : silver/{nom}")
rejets.write.mode("overwrite").parquet(f"{DOSSIER_QUALITY}/rejets_store_sales")
journal(f"Quarantaine écrite : quality/rejets_store_sales ({nb_rejets} lignes)")

# Contrôles après écriture : présence, lisibilité, nombre de lignes, schéma
controle_silver = []
for nom, df in silver.items():
    relu = spark.read.parquet(f"{DOSSIER_SILVER}/{nom}")
    controle_silver.append({"jeu": f"silver/{nom}", "lignes_ecrites": df.count(), "lignes_relues": relu.count(),
                            "colonnes": len(relu.columns),
                            "schema_conserve": [(f.name, f.dataType) for f in relu.schema] == [(f.name, f.dataType) for f in df.schema]})
pd.DataFrame(controle_silver)

15:31:07 | Silver écrit : silver/store_sales


15:31:09 | Silver écrit : silver/item
15:31:10 | Silver écrit : silver/store


15:31:12 | Silver écrit : silver/date_dim


15:31:14 | Quarantaine écrite : quality/rejets_store_sales (0 lignes)


,jeu,lignes_ecrites,lignes_relues,colonnes,schema_conserve
0,silver/store_sales,667579,667579,26,True
1,silver/item,17820,17820,25,True
2,silver/store,12,12,32,True
3,silver/date_dim,109573,109573,31,True


## 6. Jointures vérifiées

On enrichit les ventes Silver avec la **date**, l'**article** et le **magasin**.

| Jointure | Colonnes | Relation | Cardinalité attendue |
|---|---|---|---|
| ventes → date_dim | `ss_sold_date_sk` = `d_date_sk` | une vente a lieu un jour | N:1 |
| ventes → item | `ss_item_sk` = `i_item_sk` | une vente porte sur un article | N:1 |
| ventes → store | `ss_store_sk` = `s_store_sk` | une vente a lieu dans un magasin | N:1 |

Pour une relation **N:1** avec une clé unique côté référentiel, le **nombre de lignes doit rester identique** avant et après la jointure. S'il augmentait, des ventes auraient été dupliquées, ce qui fausserait le CA.

On n'utilise pas de jointure entre les ventes et la navigation web : aucun identifiant fiable ne relie ces données.

In [24]:
s_sales = spark.read.parquet(f"{DOSSIER_SILVER}/store_sales")
s_date  = spark.read.parquet(f"{DOSSIER_SILVER}/date_dim").select("d_date_sk", "d_date", "d_year", "d_moy")
s_item  = spark.read.parquet(f"{DOSSIER_SILVER}/item").select("i_item_sk", "i_item_id", "i_category", "i_class", "i_brand")
s_store = spark.read.parquet(f"{DOSSIER_SILVER}/store").select("s_store_sk", "s_store_id", "s_store_name", "s_city", "s_state")

controle_jointures = []
ventes = s_sales
for nom, ref, cle_vente, cle_ref in [("date_dim", s_date, "ss_sold_date_sk", "d_date_sk"),
                                     ("item", s_item, "ss_item_sk", "i_item_sk"),
                                     ("store", s_store, "ss_store_sk", "s_store_sk")]:
    avant = ventes.count()
    ventes = ventes.join(ref, ventes[cle_vente] == ref[cle_ref], "left")
    apres = ventes.count()
    non_rapproches = ventes.filter(F.col(cle_ref).isNull()).count()
    controle_jointures.append({"jointure": f"store_sales -> {nom}", "cle": f"{cle_vente} = {cle_ref}",
                               "lignes_avant": avant, "lignes_apres": apres,
                               "non_rapprochees": non_rapproches,
                               "statut": "OK" if avant == apres and non_rapproches == 0 else "À analyser"})
ventes = ventes.cache()
pd.DataFrame(controle_jointures)

,jointure,cle,lignes_avant,lignes_apres,non_rapprochees,statut
0,store_sales -> date_dim,ss_sold_date_sk = d_date_sk,667579,667579,0,OK
1,store_sales -> item,ss_item_sk = i_item_sk,667579,667579,0,OK
2,store_sales -> store,ss_store_sk = s_store_sk,667579,667579,0,OK


## 7. Agrégations → zone Gold

Définitions reprises du dictionnaire des KPI du **Jour 1** :
- **KPI-01 CA net** = somme de `ss_net_paid` (montant payé, hors taxes, après remises) ;
- **KPI-02 Quantités vendues** = somme de `ss_quantity` ;
- **KPI-03 Transactions** = nombre de tickets distincts (`ss_ticket_number`) ;
- **KPI-04 Panier moyen** = CA net ÷ nombre de tickets.

### Gold 1 : `ventes_magasin_mois` (question Q06)

| Élément | Description |
|---|---|
| Finalité | Comparer la performance des magasins et suivre leur évolution mensuelle |
| Sources | `silver/store_sales`, `silver/store`, `silver/date_dim` |
| Granularité | **une ligne = un magasin × un mois** |
| Dimensions | magasin (id, nom, ville, état), année, mois |
| Mesures | CA net, quantités, nombre de tickets, nombre de lignes, panier moyen |
| Limites | Données synthétiques ; ventes magasin uniquement (pas le web) ; retours non déduits ; l'année 2005 est incomplète (ventes jusqu'au 13/12/2005 dans le lot initial) |

In [25]:
gold_magasin = (ventes
    .groupBy("s_store_sk", "s_store_id", "s_store_name", "s_city", "s_state",
             F.col("d_year").alias("annee"), F.col("d_moy").alias("mois"))
    .agg(F.sum("ss_net_paid").alias("ca_net"),
         F.sum("ss_quantity").alias("quantite_vendue"),
         F.countDistinct("ss_ticket_number").alias("nb_tickets"),
         F.count("*").alias("nb_lignes_vente"))
    .withColumn("panier_moyen", F.round(F.col("ca_net") / F.col("nb_tickets"), 2))
    .withColumn("_sources", F.lit("silver/store_sales + silver/store + silver/date_dim"))
    .withColumn("_date_traitement", F.lit(DATE_TRAITEMENT))
    .orderBy("annee", "mois", "s_store_sk"))

gold_magasin.write.mode("overwrite").partitionBy("annee").parquet(f"{DOSSIER_GOLD}/ventes_magasin_mois")
journal("Gold écrit : gold/ventes_magasin_mois (partitionné par année)")
gold_magasin.show(12, truncate=False)

15:31:58 | Gold écrit : gold/ventes_magasin_mois (partitionné par année)
+----------+----------------+------------+------------+-------+-----+----+----------+---------------+----------+---------------+------------+---------------------------------------------------+-------------------+
|s_store_sk|s_store_id      |s_store_name|s_city      |s_state|annee|mois|ca_net    |quantite_vendue|nb_tickets|nb_lignes_vente|panier_moyen|_sources                                           |_date_traitement   |
+----------+----------------+------------+------------+-------+-----+----+----------+---------------+----------+---------------+------------+---------------------------------------------------+-------------------+
|0         |AAAAAAAAAAAAAAAA|Forest      |Post Oak    |CO     |2001 |1   |1386332.96|44609          |111       |861            |12489.49    |silver/store_sales + silver/store + silver/date_dim|2026-11-18 15:02:17|
|1         |AAAAAAAAAAAAAAAB|Hillside    |Jerusalem   |TN     |2001 |1 

**Partitionnement :** `ventes_magasin_mois` est rangé par année (`annee=2001/`, `annee=2002/`…). Une analyse portant sur une seule année ne lit que son dossier. On ne partitionne pas aussi par mois : cela créerait une soixantaine de très petits fichiers pour seulement quelques centaines de lignes.

### Gold 2 : `ventes_categorie_annee` (question Q03)

| Élément | Description |
|---|---|
| Finalité | Identifier les catégories qui génèrent le plus de CA et de volume |
| Sources | `silver/store_sales`, `silver/item`, `silver/date_dim` |
| Granularité | **une ligne = une catégorie × une année** |
| Mesures | CA net, quantités, nombre de lignes, part du CA de l'année (%) |
| Limites | Catégorie de l'article telle qu'enregistrée dans `item` ; ventes magasin uniquement ; données synthétiques |

In [26]:
fenetre_annee = Window.partitionBy("annee")
gold_categorie = (ventes
    .groupBy("i_category", F.col("d_year").alias("annee"))
    .agg(F.sum("ss_net_paid").alias("ca_net"),
         F.sum("ss_quantity").alias("quantite_vendue"),
         F.count("*").alias("nb_lignes_vente"))
    .withColumn("part_ca_annee_pct",
                F.round(100 * F.col("ca_net") / F.sum("ca_net").over(fenetre_annee), 2))
    .withColumn("_sources", F.lit("silver/store_sales + silver/item + silver/date_dim"))
    .withColumn("_date_traitement", F.lit(DATE_TRAITEMENT))
    .orderBy("annee", F.desc("ca_net")))

gold_categorie.write.mode("overwrite").parquet(f"{DOSSIER_GOLD}/ventes_categorie_annee")
journal("Gold écrit : gold/ventes_categorie_annee")
gold_categorie.filter(F.col("annee") == 2004).show(truncate=False)

15:32:00 | Gold écrit : gold/ventes_categorie_annee
+------------------------+-----+------------+---------------+---------------+-----------------+--------------------------------------------------+-------------------+
|i_category              |annee|ca_net      |quantite_vendue|nb_lignes_vente|part_ca_annee_pct|_sources                                          |_date_traitement   |
+------------------------+-----+------------+---------------+---------------+-----------------+--------------------------------------------------+-------------------+
|Music                   |2004 |106452534.79|3114031        |61358          |45.43            |silver/store_sales + silver/item + silver/date_dim|2026-11-18 15:02:17|
|Books                   |2004 |62720328.90 |1838029        |36211          |26.76            |silver/store_sales + silver/item + silver/date_dim|2026-11-18 15:02:17|
|Movies & TV             |2004 |15029271.65 |446264         |8835           |6.41             |silver/store_sales

### Contrôle de réconciliation

Le CA total de chaque jeu Gold doit être **égal** au CA de Silver. Si aucune ligne n'a été écartée, il doit aussi être égal au **KPI-01 calculé au Jour 1** (1 150 529 586,37). C'est un contrôle **indépendant** : il compare deux calculs faits avec deux outils différents.

In [27]:
KPI01_JOUR1 = 1150529586.37
ca_silver = float(s_sales.agg(F.sum("ss_net_paid")).first()[0])
ca_gold_mag = float(spark.read.parquet(f"{DOSSIER_GOLD}/ventes_magasin_mois").agg(F.sum("ca_net")).first()[0])
ca_gold_cat = float(spark.read.parquet(f"{DOSSIER_GOLD}/ventes_categorie_annee").agg(F.sum("ca_net")).first()[0])
tickets_gold = spark.read.parquet(f"{DOSSIER_GOLD}/ventes_magasin_mois").agg(F.sum("nb_tickets")).first()[0]

reconciliation = pd.DataFrame([
    ("CA net Silver", round(ca_silver, 2)),
    ("CA net Gold magasin × mois", round(ca_gold_mag, 2)),
    ("CA net Gold catégorie × année", round(ca_gold_cat, 2)),
    ("KPI-01 calculé au Jour 1", KPI01_JOUR1),
    ("Écart Gold / Silver", round(ca_gold_mag - ca_silver, 2)),
    ("Écart Silver / Jour 1", round(ca_silver - KPI01_JOUR1, 2)),
    ("Tickets (somme Gold) / tickets distincts Silver", f"{tickets_gold} / {s_sales.select('ss_ticket_number').distinct().count()}"),
], columns=["Contrôle", "Valeur"])
reconciliation

,Contrôle,Valeur
0,CA net Silver,1150529586.37
1,CA net Gold magasin × mois,1150529586.37
2,CA net Gold catégorie × année,1150529586.37
3,KPI-01 calculé au Jour 1,1150529586.37
4,Écart Gold / Silver,0.0
5,Écart Silver / Jour 1,0.0
6,Tickets (somme Gold) / tickets distincts Silver,89100 / 89100


## 8. Parquet : écriture, relecture et comparaison des formats

On relit chaque jeu Gold pour vérifier que la **lecture fonctionne**, que le **schéma** et le **nombre de lignes** sont conservés et que les valeurs sont exploitables.

In [28]:
for nom in ["ventes_magasin_mois", "ventes_categorie_annee"]:
    relu = spark.read.parquet(f"{DOSSIER_GOLD}/{nom}")
    print(f"\n=== gold/{nom} : {relu.count()} lignes relues ===")
    relu.printSchema()


=== gold/ventes_magasin_mois : 720 lignes relues ===
root
 |-- s_store_sk: long (nullable = true)
 |-- s_store_id: string (nullable = true)
 |-- s_store_name: string (nullable = true)
 |-- s_city: string (nullable = true)
 |-- s_state: string (nullable = true)
 |-- mois: integer (nullable = true)
 |-- ca_net: decimal(22,2) (nullable = true)
 |-- quantite_vendue: long (nullable = true)
 |-- nb_tickets: long (nullable = true)
 |-- nb_lignes_vente: long (nullable = true)
 |-- panier_moyen: decimal(23,2) (nullable = true)
 |-- _sources: string (nullable = true)
 |-- _date_traitement: string (nullable = true)
 |-- annee: integer (nullable = true)


=== gold/ventes_categorie_annee : 45 lignes relues ===
root
 |-- i_category: string (nullable = true)
 |-- annee: integer (nullable = true)
 |-- ca_net: decimal(22,2) (nullable = true)
 |-- quantite_vendue: long (nullable = true)
 |-- nb_lignes_vente: long (nullable = true)
 |-- part_ca_annee_pct: decimal(29,2) (nullable = true)
 |-- _sources: s

### Comparaison CSV / JSON / Parquet

On compare les formats sur **les mêmes données** :
- la **source brute** `store_sales.dat` (texte délimité, équivalent d'un CSV) contre sa version **Silver en Parquet** ;
- le jeu Gold magasin × mois écrit en **CSV**, en **JSON** et en **Parquet**.

In [29]:
def taille_dossier(chemin):
    if os.path.isfile(chemin):
        return os.path.getsize(chemin)
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(chemin)
               for f in fs if not f.startswith((".", "_")))

comp = f"{DOSSIER_OUTPUTS}/comparaison_formats"
g = spark.read.parquet(f"{DOSSIER_GOLD}/ventes_magasin_mois").coalesce(1)
g.write.mode("overwrite").option("header", True).csv(f"{comp}/csv")
g.write.mode("overwrite").json(f"{comp}/json")
g.write.mode("overwrite").parquet(f"{comp}/parquet")

tailles = pd.DataFrame([
    ("store_sales : source .dat (texte)", taille_dossier(f"{DOSSIER_RAW}/store_sales.dat")),
    ("store_sales : Silver Parquet", taille_dossier(f"{DOSSIER_SILVER}/store_sales")),
    ("Gold magasin × mois : CSV", taille_dossier(f"{comp}/csv")),
    ("Gold magasin × mois : JSON", taille_dossier(f"{comp}/json")),
    ("Gold magasin × mois : Parquet", taille_dossier(f"{comp}/parquet")),
], columns=["Jeu de données", "Taille (octets)"])
tailles["Taille (Mo)"] = (tailles["Taille (octets)"] / 1024 / 1024).round(2)
tailles.to_csv(f"{DOSSIER_OUTPUTS}/comparaison_tailles.csv", index=False)
tailles

,Jeu de données,Taille (octets),Taille (Mo)
0,store_sales : source .dat (texte),92764700,88.47
1,store_sales : Silver Parquet,61826532,58.96
2,Gold magasin × mois : CSV,110925,0.11
3,Gold magasin × mois : JSON,249015,0.24
4,Gold magasin × mois : Parquet,20183,0.02


**Lecture sélective :** avec Parquet, Spark ne lit que les colonnes demandées. Le plan d'exécution ci-dessous le montre : la ligne `ReadSchema` ne contient que les colonnes utilisées.

In [30]:
spark.read.parquet(f"{DOSSIER_SILVER}/store_sales").select("ss_store_sk", "ss_net_paid").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [ss_store_sk#17233L,ss_net_paid#17246] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/mnt/c/Users/found/desktop/ipssi/bd/retail_project/J3/tp3/silver/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<ss_store_sk:bigint,ss_net_paid:decimal(12,2)>




| Critère | CSV / `.dat` | JSON | Parquet |
|---|---|---|---|
| Organisation | Lignes, valeurs séparées par un délimiteur | Un objet par ligne (clé : valeur) | **Colonnes** regroupées, avec métadonnées |
| Lisibilité | Lisible dans un éditeur | Lisible | **Binaire** (non lisible à l'œil) |
| Types | Aucun : tout est du texte, à réinterpréter | Types simples (nombre, texte, booléen) | **Schéma typé** conservé (décimal, date…) |
| Volume observé | voir tableau ci-dessus | le plus volumineux (noms de colonnes répétés) | le plus compact (compression par colonne) |
| Lecture sélective | Non : il faut lire toute la ligne | Non | **Oui**, colonne par colonne |
| Usages adaptés | Échange, export, sources brutes (Raw) | Données semi-structurées, API | **Analyse**, zones Silver et Gold |
| Limites | Pas de types, séparateurs fragiles | Volumineux, lent à analyser | Illisible sans outil ; **pas de transactions ACID** à lui seul |

⚠️ Les fichiers Parquet produits ici **ne sont pas des tables transactionnelles Lakehouse**. Les garanties ACID, l'historique des versions et l'évolution contrôlée du schéma nécessitent un **format de table** (Delta Lake, Apache Iceberg ou Apache Hudi), qui n'est pas utilisé dans ce TP.

## 9. Rapport qualité

Synthèse de tous les contrôles, au format **contrôle / règle / résultat / statut**. Le rapport est enregistré dans `quality/rapport_qualite.csv`.

In [31]:
def statut(ok, alerte=False):
    return "OK" if ok else ("À surveiller" if alerte else "KO")

vm_ss = explorations["store_sales"].set_index("colonne")["nb_nulls"]
oblig = ["ss_sold_date_sk", "ss_item_sk", "ss_store_sk", "ss_ticket_number"]
rej = {r["motif_rejet"]: r["count"] for r in rejets.groupBy("motif_rejet").count().collect()}
cnt_ref = {t: typees[t].count() for t in ["item", "store", "date_dim"]}

C = []
def ajout(jeu, dimension, controle, regle, resultat, st):
    C.append({"jeu": jeu, "dimension": dimension, "controle": controle, "regle": regle, "resultat": resultat, "statut": st})

ajout("sources", "Volume", "Nombre de lignes lues", "Identique à l'inventaire du Jour 1",
      f"store_sales {nb_initial}, item {cnt_ref['item']}, store {cnt_ref['store']}, date_dim {cnt_ref['date_dim']}",
      statut(nb_initial == 667579 and cnt_ref["item"] == 17820 and cnt_ref["store"] == 12 and cnt_ref["date_dim"] == 109573))
nb_echecs = sum(sum(v.values()) for v in echecs_conversion.values())
ajout("toutes", "Validité", "Conversion des types (R1)", "0 valeur non convertible", f"{nb_echecs} échec(s)", statut(nb_echecs == 0))
ajout("store_sales", "Complétude", "Champs obligatoires non nuls (R2)", "date, article, magasin, ticket renseignés",
      f"{int(sum(vm_ss[c] for c in oblig))} valeur(s) manquante(s)", statut(sum(vm_ss[c] for c in oblig) == 0))
fac = vm_ss[[c for c in ["ss_customer_sk", "ss_cdemo_sk", "ss_hdemo_sk", "ss_addr_sk", "ss_promo_sk"]]]
ajout("store_sales", "Complétude", "Champs facultatifs (R3)", "Valeurs manquantes conservées",
      ", ".join(f"{c} {int(v)}" for c, v in fac.items()), "OK (documenté)")
ajout("store_sales", "Unicité", "Doublons exacts (R4)", "0 doublon après nettoyage", f"{nb_doublons_exacts} retiré(s)", "OK")
ajout("store_sales", "Unicité", "Clé ticket + article (R5)", "Paire unique, sinon analyse (pas de suppression)",
      f"{nb_doublons_grain} ligne(s) en plus, conservées (quantités et prix différents)", statut(nb_doublons_grain == 0, True))
ajout("store_sales", "Validité", "Quantité > 0 (R6)", "ss_quantity > 0", f"{rej.get('quantite_invalide', 0)} ligne(s) en quarantaine",
      statut(rej.get("quantite_invalide", 0) == 0, True))
ajout("store_sales", "Validité", "Montants positifs (R7)", "ss_sales_price ≥ 0 et ss_net_paid ≥ 0",
      f"{rej.get('montant_negatif', 0)} ligne(s) en quarantaine", statut(rej.get("montant_negatif", 0) == 0, True))
ajout("store_sales", "Cohérence", "Marge négative (R8)", "Conservée et signalée",
      f"{nb_marge_negative} lignes ({100*nb_marge_negative/nb_silver:.1f} %)", "À surveiller")
ajout("store_sales", "Cohérence", "Calcul des montants (R9)", "ext_sales_price = quantité × prix (±0,01)",
      f"{nb_incoherence_calcul} écart(s)", statut(nb_incoherence_calcul == 0))
for m, lib in [("article_inconnu", "article → item"), ("magasin_inconnu", "magasin → store"), ("date_inconnue", "date → date_dim")]:
    ajout("store_sales", "Intégrité", f"Clé {lib} (R10)", "Chaque clé existe dans le référentiel",
          f"{rej.get(m, 0)} orphelin(s)", statut(rej.get(m, 0) == 0, True))
ajout("item", "Unicité", "Clé i_item_sk", "Unique", "unique", "OK")
ajout("item", "Complétude", "Catégorie renseignée (R12)", "Absente → « Non renseignée »", f"{nb_categorie_absente} remplacée(s)",
      statut(nb_categorie_absente == 0, True))
ajout("store", "Unicité", "Clé s_store_sk", "Unique", "unique", "OK")
ajout("date_dim", "Cohérence", "d_year et d_moy cohérents avec d_date (R13)", "Même année et même mois",
      f"{nb_dates_incoherentes} incohérence(s)", statut(nb_dates_incoherentes == 0))
ajout("silver/store_sales", "Volume", "Conservation des lignes", "initial = doublons + quarantaine + Silver",
      f"{nb_initial} = {nb_doublons_exacts} + {nb_rejets} + {nb_silver}", statut(nb_initial == nb_doublons_exacts + nb_rejets + nb_silver))
for cj in controle_jointures:
    ajout("jointures", "Intégrité", cj["jointure"], "Lignes avant = après, 0 non rapprochée",
          f"{cj['lignes_avant']} → {cj['lignes_apres']}, {cj['non_rapprochees']} non rapprochée(s)", cj["statut"])
ajout("gold", "Cohérence", "Réconciliation du CA", "CA Gold = CA Silver", f"écart {ca_gold_mag - ca_silver:.2f}",
      statut(abs(ca_gold_mag - ca_silver) < 0.01 and abs(ca_gold_cat - ca_silver) < 0.01))
ajout("gold", "Cohérence", "Comparaison avec le Jour 1", "CA Silver = KPI-01 du Jour 1 (si 0 rejet)",
      f"écart {ca_silver - KPI01_JOUR1:.2f}", statut(abs(ca_silver - KPI01_JOUR1) < 0.01, True))
ajout("gold", "Fraîcheur", "Date du traitement", "Tracée dans chaque jeu (_date_traitement)", DATE_TRAITEMENT, "OK")

rapport_qualite = pd.DataFrame(C)
rapport_qualite.to_csv(f"{DOSSIER_QUALITY}/rapport_qualite.csv", index=False)
journal("Rapport qualité écrit : quality/rapport_qualite.csv")
pd.set_option("display.max_colwidth", 80)
rapport_qualite

15:32:18 | Rapport qualité écrit : quality/rapport_qualite.csv


,jeu,dimension,controle,regle,resultat,statut
0,sources,Volume,Nombre de lignes lues,Identique à l'inventaire du Jour 1,"store_sales 667579, item 17820, store 12, date_dim 109573",OK
1,toutes,Validité,Conversion des types (R1),0 valeur non convertible,0 échec(s),OK
2,store_sales,Complétude,Champs obligatoires non nuls (R2),"date, article, magasin, ticket renseignés",0 valeur(s) manquante(s),OK
3,store_sales,Complétude,Champs facultatifs (R3),Valeurs manquantes conservées,"ss_customer_sk 0, ss_cdemo_sk 238, ss_hdemo_sk 295, ss_addr_sk 448, ss_promo...",OK (documenté)
4,store_sales,Unicité,Doublons exacts (R4),0 doublon après nettoyage,0 retiré(s),OK
5,store_sales,Unicité,Clé ticket + article (R5),"Paire unique, sinon analyse (pas de suppression)","4136 ligne(s) en plus, conservées (quantités et prix différents)",À surveiller
6,store_sales,Validité,Quantité > 0 (R6),ss_quantity > 0,0 ligne(s) en quarantaine,OK
7,store_sales,Validité,Montants positifs (R7),ss_sales_price ≥ 0 et ss_net_paid ≥ 0,0 ligne(s) en quarantaine,OK
8,store_sales,Cohérence,Marge négative (R8),Conservée et signalée,495971 lignes (74.3 %),À surveiller
9,store_sales,Cohérence,Calcul des montants (R9),"ext_sales_price = quantité × prix (±0,01)",0 écart(s),OK


### Bilan qualité

> 📝 *À compléter à partir du tableau :*
> - **règles respectées** : … ;
> - **anomalies restantes** : par exemple la marge négative, conservée et signalée ;
> - **données mises à l'écart** : nombre de lignes en quarantaine et motifs ;
> - **limites des contrôles** : données synthétiques ; contrôles réalisés sur le lot `data/` uniquement (le lot `data_refresh` n'est pas traité) ; pas de contrôle de fraîcheur réel (pas de date de chargement source) ;
> - **corrections éventuelles** : …

## 10. Sauvegarde et conclusion

Dans cette exécution locale sous WSL, les résultats sont conservés dans `retail_project/resultats/`, à côté des dossiers `data/`, `J1/`, `J2/` et `J3/`. Les sources originales de `data/` restent inchangées.


In [32]:
with open(f"{DOSSIER_OUTPUTS}/journal_execution.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(JOURNAL))

destination = os.path.join(DOSSIER_DRIVE, "resultats")
for dossier in ["silver", "gold", "quality", "outputs"]:
    shutil.copytree(f"{DOSSIER_TRAVAIL}/{dossier}", f"{destination}/{dossier}", dirs_exist_ok=True)
journal(f"Résultats copiés vers {destination}")

for racine, dossiers, fichiers in os.walk(destination):
    niveau = racine.replace(destination, "").count(os.sep)
    if niveau <= 2:
        print("  " * niveau + os.path.basename(racine) + "/")

15:32:33 | Résultats copiés vers /mnt/c/Users/found/desktop/ipssi/bd/retail_project/J3/resultats
resultats/
  gold/
    ventes_categorie_annee/
    ventes_magasin_mois/
  outputs/
    comparaison_formats/
    test_ecriture/
  quality/
    rejets_store_sales/
  silver/
    date_dim/
    item/
    store/
    store_sales/


### Conclusion

> 📝 *À compléter :* ce que la chaîne Raw → Silver → Gold a produit, ce que montrent les jeux Gold pour Q06 et Q03 (magasin le plus performant, catégorie dominante), et les limites (données synthétiques, mode local, repli sur Google Drive au lieu de HDFS).

Ces jeux Gold serviront aux analyses et visualisations du **Jour 4**.

In [39]:
spark.stop()
print("Session Spark arrêtée.")

Session Spark arrêtée.
